In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
train_part, valid_part = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["answer"]
)

print(train_part.shape)
print(valid_part.shape)

(1600, 8)
(400, 8)


In [5]:
X_train = []
y_train = []

for _, row in train_part.iterrows():

    for option in ["A","B","C","D","E"]:

        text = row["prompt"] + " " + str(row[option])

        X_train.append(text)

        y_train.append(
            1 if row["answer"] == option else 0
        )

print(len(X_train))
print(len(y_train))

8000
8000


In [6]:
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)

print(X_train_vec.shape)

(8000, 28429)


In [7]:
X_train_tensor = torch.FloatTensor(
    X_train_vec.toarray()
)

y_train_tensor = torch.LongTensor(y_train)

print(X_train_tensor.shape)
print(y_train_tensor.shape)


torch.Size([8000, 28429])
torch.Size([8000])


In [8]:
class MCQNet(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 512),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(512,128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128,2)
        )
    def forward(self,x):

        return self.network(x)

In [9]:
input_size = X_train_tensor.shape[1]

model = MCQNet(input_size)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

MCQNet(
  (network): Sequential(
    (0): Linear(in_features=28429, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [10]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_train_tensor)

    loss = criterion(
        outputs,
        y_train_tensor
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}/{epochs}  Loss: {loss.item():.4f}"
    )

Epoch 1/5  Loss: 0.7056
Epoch 2/5  Loss: 0.6927
Epoch 3/5  Loss: 0.6792
Epoch 4/5  Loss: 0.6635
Epoch 5/5  Loss: 0.6453


In [11]:
X_valid = []
y_valid = []

for _, row in valid_part.iterrows():

    for option in ["A","B","C","D","E"]:

        X_valid.append(
            row["prompt"] + " " + str(row[option])
        )

        y_valid.append(
            1 if row["answer"] == option else 0
        )

print(len(X_valid))

2000


In [12]:
X_valid_vec = vectorizer.transform(X_valid)

X_valid_tensor = torch.FloatTensor(
    X_valid_vec.toarray()
)

print(X_valid_tensor.shape)

torch.Size([2000, 28429])


In [13]:
model.eval()

with torch.no_grad():

    outputs = model(X_valid_tensor)

    probs = torch.softmax(outputs, dim=1)[:,1].numpy()

print(probs[:10])

[0.46151394 0.46077073 0.47110873 0.46123117 0.46063083 0.45113865
 0.4442799  0.44893557 0.48735857 0.45063907]


In [14]:
predictions = []

idx = 0

for _, row in valid_part.iterrows():

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        scores[option] = probs[idx]

        idx += 1

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(ranked[:3])

print(predictions[:5])

[['C', 'A', 'D'], ['D', 'A', 'E'], ['E', 'C', 'A'], ['D', 'E', 'A'], ['B', 'D', 'C']]


In [15]:
def map3(actual, predicted):

    score = 0.0

    for truth, preds in zip(actual, predicted):

        for rank, pred in enumerate(preds[:3]):

            if pred == truth:

                score += 1 / (rank + 1)

                break

    return score / len(actual)

In [16]:
actual = valid_part["answer"].tolist()

score = map3(actual, predictions)

print(f"Validation MAP@3: {score:.5f}")

Validation MAP@3: 0.93875


In [17]:
X_full = []
y_full = []

for _, row in train.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        X_full.append(
            row["prompt"] + " " + str(row[option])
        )

        y_full.append(
            1 if row["answer"] == option else 0
        )

In [18]:
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_full_vec = vectorizer.fit_transform(X_full)

X_full_tensor = torch.FloatTensor(
    X_full_vec.toarray()
)

y_full_tensor = torch.LongTensor(y_full)

model = MCQNet(X_full_tensor.shape[1])

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [19]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_full_tensor)

    loss = criterion(outputs, y_full_tensor)

    loss.backward()

    optimizer.step()

    print(f"Epoch {epoch+1}: {loss.item():.4f}")


Epoch 1: 0.6570
Epoch 2: 0.6421
Epoch 3: 0.6239
Epoch 4: 0.6035
Epoch 5: 0.5793


In [20]:
X_test = []

for _, row in test.iterrows():

    for option in ["A","B","C","D","E"]:

        X_test.append(
            row["prompt"] + " " + str(row[option])
        )

In [21]:
X_test_vec = vectorizer.transform(X_test)

X_test_tensor = torch.FloatTensor(
    X_test_vec.toarray()
)

In [22]:
model.eval()

with torch.no_grad():

    outputs = model(X_test_tensor)

    probs = torch.softmax(outputs, dim=1)[:,1].numpy()

In [23]:
predictions = []

idx = 0

for _, row in test.iterrows():

    scores = {}

    for option in ["A","B","C","D","E"]:

        scores[option] = probs[idx]
        idx += 1

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(ranked[:3])

In [24]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(p) for p in predictions]
})

submission.head()


,ID,Prediction
0,1,A E B
1,2,B E A
2,3,B E A
3,4,E C D
4,5,C A B


In [25]:
submission.to_csv(
    "submission_neural.csv",
    index=False
)

print("Submission saved!")

Submission saved!


In [26]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission_neural.csv']


In [27]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

import os
print(os.path.exists("/kaggle/working/submission.csv"))

True
